In [1]:
import re
import nltk

emotions_str = r"""
    (?:
        [:=;] # 眼睛
        [oO\-]? # ⿐鼻⼦子
        [D\)\]\(\]/\\OpP] # 嘴
    )"""
regex_str = [
    emotions_str,
    r'<[^>]+>', # HTML tags
    r'(?:@[\w_]+)', # @某⼈人
    r"(?:\#+[\w_]+[\w\'_\-]*[\w_]+)", # 话题标签
    r'http[s]?://(?:[a-z]|[0-9]|[$-_@.&amp;+]|[!*\(\),]|(?:%[0-9a-f][0-9a-f]))+',
    # URLs
    r'(?:(?:\d+,?)+(?:\.?\d+)?)', # 数字
    r"(?:[a-z][a-z'\-_]+[a-z])", # 含有 - 和 ‘ 的单词
    r'(?:[\w_]+)', # 其他
    r'(?:\S)' # 其他
    ]

tokens_re = re.compile(r'('+'|'.join(regex_str)+')',re.VERBOSE | re.IGNORECASE)#所有特殊字符的匹配式
emotion_re = re.compile(r'^'+emotions_str+'$',re.VERBOSE | re.IGNORECASE)#表情符号的匹配式

def tokenize(s):
    return tokens_re.findall(s) #匹配出所有特殊字符

def preprocess(s,lowercase=False):
    tokens = tokenize(s)
    if lowercase:
        tokens = [token if emotion_re.search(token) else token.lower() for token in tokens]#对非表情符号进行小写处理
    return tokens#返回所有特殊字符

tweet = 'RT @angelababy: love you baby! :D http://ah.love #168cm'
print(preprocess(tweet))

['RT', '@angelababy', ':', 'love', 'you', 'baby', '!', ':D', 'http://ah.love', '#168cm']


In [10]:
from nltk.tokenize import word_tokenize
tweet = 'RT @angelababy: love you baby! :D http://ah.love #168cm'
print(word_tokenize(tweet))

['RT', '@', 'angelababy', ':', 'love', 'you', 'baby', '!', ':', 'D', 'http', ':', '//ah.love', '#', '168cm']


In [11]:
import numpy as np

a = np.array([(1,2.5,4), (2,3.1,5)], dtype=[('a', 'i4'),('b', 'f4'),('c', 'i8')])
np.save("a.npy", a)

### 提取词干

In [12]:
from nltk.stem.porter import PorterStemmer
from nltk.stem import LancasterStemmer

porter_stemmer = PorterStemmer()
porter_stemmer.stem('maximum')
lancaster_stemmer = LancasterStemmer()
lancaster_stemmer.stem('maximum')

'maxim'

In [13]:
from nltk.stem import WordNetLemmatizer as WL
wl = WL()
wl.lemmatize('dogs')


'dog'

有时候同样的单词会有不同意思在句子中担任不同的成分
如 went 动词的时候 是go 的过去式， 名词的时候为人名 温特
因此我们需要引入 POS TAG （Part-Of-Speech)

<img src='https://pic3.zhimg.com/80/v2-41444364cf4c5fbb7b44c9540aa99126_hd.jpg'>

In [14]:
wl.lemmatize('is'),wl.lemmatize('is',pos='v')


('is', 'be')

NLTK 标注POS TAG

In [15]:
text = nltk.word_tokenize('what does the fox say')
text

['what', 'does', 'the', 'fox', 'say']

In [16]:
nltk.pos_tag(text)

[('what', 'WDT'),
 ('does', 'VBZ'),
 ('the', 'DT'),
 ('fox', 'NNS'),
 ('say', 'VBP')]

Stopwords 停用词

⼀千个HE有⼀千种指代

⼀千个THE有⼀千种指事

对于注重理解⽂本『意思』的应⽤场景来说

歧义太多

In [17]:
from nltk.corpus import stopwords
# 先token⼀一把，得到⼀一个word_list
# ...
# 然后filter⼀一把
text = '''
what does the fox say
'''
filtered_words = [word for word in text if word not in stopwords.words('english')]
filtered_words

['\n', 'w', 'h', ' ', 'e', ' ', 'h', 'e', ' ', 'f', 'x', ' ', '\n']

### 情感分析

In [18]:
from nltk.classify import NaiveBayesClassifier
# 随⼿手造点训练集
s1 = 'this is a good book'
s2 = 'this is a awesome book'
s3 = 'this is a bad book'
s4 = 'this is a terrible book'
def proprocess(s):
    # Func: 句句⼦子处理理
    # 这⾥里里简单的⽤用了了split(), 把句句⼦子中每个单词分开
    # 显然 还有更更多的processing method可以⽤用
    return {word: True for word in s.lower().split() }
    # return⻓长这样:
    # {'this': True, 'is':True, 'a':True, 'good':True, 'book':True}
    # 其中, 前⼀一个叫fname, 对应每个出现的⽂文本单词;
    # 后⼀一个叫fval, 指的是每个⽂文本单词对应的值。
    # 这⾥里里我们⽤用最简单的True,来表示,这个词『出现在当前的句句⼦子中』的意义。
    # 当然啦, 我们以后可以升级这个⽅方程, 让它带有更更加⽜牛逼的fval, ⽐比如 word2vec

# 把训练集给做成标准形式
training_data = [[proprocess(s1),'pos'],
                [proprocess(s2),'pos'],
                [proprocess(s3),'neg'],
                [proprocess(s4),'neg']]
training_data

[[{'this': True, 'is': True, 'a': True, 'good': True, 'book': True}, 'pos'],
 [{'this': True, 'is': True, 'a': True, 'awesome': True, 'book': True}, 'pos'],
 [{'this': True, 'is': True, 'a': True, 'bad': True, 'book': True}, 'neg'],
 [{'this': True, 'is': True, 'a': True, 'terrible': True, 'book': True},
  'neg']]

In [19]:
# 喂给model吃
model = NaiveBayesClassifier.train(training_data)
print(model.classify(proprocess('this is a good book')))

pos


其实就是通过训练集进行训练 然后对目标测试集进行结果分析



看上边的proprocess的函数其实还可以更加完善，对其tokens，这里只有简单的断开split，带pos tag的lemma ，去除stopwords

In [20]:
def proprocess2(s):
    tokens2 = proprocess(s)
    filterer_words2 = [word for word in tokens2 if word not in stopwords.words('english')]
    pos_tag = nltk.pos_tag(filterer_words2)
    return {i[0]:i[1]    for i in pos_tag}
proprocess2('this is a good book')

{'good': 'JJ', 'book': 'NN'}

文本相似度

<img src="https://pic3.zhimg.com/80/v2-020d42468bc9de1585ef7e3bf490092e_hd.jpg" />
⽤元素频率表⽰⽂本特征

<img src="https://pic1.zhimg.com/80/v2-37a7c82ec4cdfba8a43c67e6342c23b4_hd.jpg" />
余弦定理

对文本进行向量化，当两个向量之间的夹角越小，就认为这两个向量相等，则这两文本相似

Frequency 频率统计

In [21]:
from nltk import FreqDist
corpus = 'this is my sentence '\
        'this is my life '\
     'this is the day'

# 随便便tokenize⼀一下
# 显然, 正如上⽂文提到,
# 这⾥里里可以根据需要做任何的preprocessing:
# stopwords, lemma, stemming, etc.
tokens = nltk.word_tokenize(corpus)
print(tokens)
# 得到token好的word list
# ['this', 'is', 'my', 'sentence',
# 'this', 'is', 'my', 'life', 'this',
# 'is', 'the', 'day']

# 借⽤用NLTK的FreqDist统计⼀一下⽂文字出现的频率
fdist = FreqDist(tokens)

# 它就类似于⼀一个Dict
# 带上某个单词, 可以看到它在整个⽂文章中出现的次数
print(fdist['is'])
# 3

# 好, 此刻, 我们可以把最常⽤用的50个单词拿出来
standard_freq_vector = fdist.most_common(50)
size = len(standard_freq_vector)
print(standard_freq_vector)
# [('is', 3), ('this', 3), ('my', 2),
# ('the', 1), ('day', 1), ('sentence', 1),
# ('life', 1)

# Func: 按照出现频率⼤大⼩小, 记录下每⼀一个单词的位置
def position_lookup(v):
    res = {}
    counter = 0
    for word in v:
        res[word[0]] = counter
        counter += 1
    return res

# 把标准的单词位置记录下来
standard_position_dict = position_lookup(standard_freq_vector)
print(standard_position_dict)
# 得到⼀一个位置对照表
# {'is': 0, 'the': 3, 'day': 4, 'this': 1,
# 'sentence': 5, 'my': 2, 'life': 6}

# 这时, 如果我们有个新句句⼦子:
sentence = 'this is cool'
# 先新建⼀一个跟我们的标准vector同样⼤大⼩小的向量量
freq_vector = [0] * size
# 简单的Preprocessing
tokens = nltk.word_tokenize(sentence)
# 对于这个新句句⼦子⾥里里的每⼀一个单词
for word in tokens:
    try:
        # 如果在我们的词库⾥里里出现过
        # 那么就在"标准位置"上+1
        freq_vector[standard_position_dict[word]] += 1
    except KeyError:
        # 如果是个新词
        # 就pass掉
        continue
    print(freq_vector)
        # [1, 1, 0, 0, 0, 0, 0]
        # 第⼀一个位置代表 is, 出现了了⼀一次
        # 第⼆二个位置代表 this, 出现了了⼀一次

['this', 'is', 'my', 'sentence', 'this', 'is', 'my', 'life', 'this', 'is', 'the', 'day']
3
[('this', 3), ('is', 3), ('my', 2), ('sentence', 1), ('life', 1), ('the', 1), ('day', 1)]
{'this': 0, 'is': 1, 'my': 2, 'sentence': 3, 'life': 4, 'the': 5, 'day': 6}
[1, 0, 0, 0, 0, 0, 0]
[1, 1, 0, 0, 0, 0, 0]


### 文本分类

TF: Term Frequency, 衡量⼀个term在⽂档中出现得有多频繁。

TF(t) = (t出现在⽂档中的次数) / (⽂档中的term总数).

IDF: Inverse Document Frequency, 衡量⼀个term有多重要。

有些词出现的很多，但是明显不是很有卵⽤。⽐如’is'，’the‘，’and‘之类的。

为了平衡，我们把罕见的词的重要性（weight）搞⾼，

把常见词的重要性搞低。

IDF(t) = log_e(⽂档总数 / 含有t的⽂档总数).

TF-IDF = TF * IDF

当然wiki百科下面解释更全面

https://zh.wikipedia.org/wiki/Tf-idf
​
zh.wikipedia.org

举个栗⼦🌰:

⼀个⽂档有100个单词，其中单词baby出现了3次。

那么，TF(baby) = (3/100) = 0.03.

好，现在我们如果有10M的⽂档， baby出现在其中的1000个⽂档中。

那么，IDF(baby) = log(10,000,000 / 1,000) = 4

所以， TF-IDF(baby) = TF(baby) * IDF(baby) = 0.03 * 4 = 0.12

In [22]:
from nltk.text import TextCollection

# ⾸首先, 把所有的⽂文档放到TextCollection类中。
# 这个类会⾃自动帮你断句句, 做统计, 做计算
corpus = TextCollection(['this is sentence one',
'this is sentence two',
'this is sentence three'])

# 直接就能算出tfidf
# (term: ⼀一句句话中的某个term, text: 这句句话)
print(corpus.tf_idf('is', 'this is sentence one'))
# 0.444342

# # 同理理, 怎么得到⼀一个标准⼤大⼩小的vector来表示所有的句句⼦子?

# # 对于每个新句句⼦子
# new_sentence = 'this is sentence five'
# # 遍历⼀一遍所有的vocabulary中的词:
# for word in corpus:
#     print(corpus.tf_idf(word, new_sentence))
#     # 我们会得到⼀一个巨⻓长(=所有vocab⻓长度)的向量量

0.0


In [23]:
from __future__ import division
import nltk
import matplotlib
from nltk.book import *
from nltk.util import bigrams

# 单词搜索
print('单词搜索')
text1.concordance('boy')
text2.concordance('friends')

# 相似词搜索
print('相似词搜索')
text3.similar('time')

#共同上下文搜索
print('共同上下文搜索')
text2.common_contexts(['monstrous','very'])

# 词汇分布表
print('词汇分布表')
text4.dispersion_plot(['citizens', 'American', 'freedom', 'duties'])

# 词汇计数
print('词汇计数')
print(len(text5))
sorted(set(text5))
print(len(set(text5)))

# 重复词密度
print('重复词密度')
print(len(text8) / len(set(text8)))

# 关键词密度
print('关键词密度')
print(text9.count('girl'))
print(text9.count('girl') * 100 / len(text9))

# 频率分布
fdist = FreqDist(text1)

vocabulary = fdist.keys()
for i in vocabulary:
    print(i)

# 高频前20
fdist.plot(20, cumulative = True)

# 低频词
print('低频词：')
print(fdist.hapaxes())

# 词语搭配
print('词语搭配')
words = list(bigrams(['louder', 'words', 'speak']))
print(words)

*** Introductory Examples for the NLTK Book ***
Loading text1, ..., text9 and sent1, ..., sent9
Type the name of the text or sentence to view it.
Type: 'texts()' or 'sents()' to list the materials.
text1: Moby Dick by Herman Melville 1851
text2: Sense and Sensibility by Jane Austen 1811
text3: The Book of Genesis
text4: Inaugural Address Corpus
text5: Chat Corpus
text6: Monty Python and the Holy Grail
text7: Wall Street Journal
text8: Personals Corpus
text9: The Man Who Was Thursday by G . K . Chesterton 1908
单词搜索
Displaying 25 of 65 matches:
 ? Why is almost every robust healthy boy with a robust healthy soul in him , a
lings in a most direful manner . " My boy ," said the landlord , " you ' ll hav
idends . Rising from a little cabin - boy in short clothes of the drabbest drab
ain ' t Captain Peleg ; HE ' S AHAB , boy ; and Ahab of old , thou knowest , wa
 to have a wicked name . Besides , my boy , he has a wife -- not three voyages 
stors , and scolding her little black boy meantime 

<Figure size 640x480 with 1 Axes>

词汇计数
45010
6066
重复词密度
4.39259927797834
关键词密度
10
0.01444815280366405
[
Moby
Dick
by
Herman
Melville
1851
]
ETYMOLOGY
.
(
Supplied
a
Late
Consumptive
Usher
to
Grammar
School
)
The
pale
--
threadbare
in
coat
,
heart
body
and
brain
;
I
see
him
now
He
was
ever
dusting
his
old
lexicons
grammars
with
queer
handkerchief
mockingly
embellished
all
the
gay
flags
of
known
nations
world
loved
dust
it
somehow
mildly
reminded
mortality
"
While
you
take
hand
school
others
teach
them
what
name
whale
-
fish
is
be
called
our
tongue
leaving
out
through
ignorance
letter
H
which
almost
alone
maketh
signification
word
deliver
that
not
true
."
HACKLUYT
WHALE
...
Sw
Dan
HVAL
This
animal
named
from
roundness
or
rolling
for
HVALT
arched
vaulted
WEBSTER
'
S
DICTIONARY
It
more
immediately
Dut
Ger
WALLEN
A
WALW
IAN
roll
wallow
RICHARDSON
KETOS
GREEK
CETUS
LATIN
WHOEL
ANGLO
SAXON
DANISH
WAL
DUTCH
HWAL
SWEDISH
ICELANDIC
ENGLISH
BALEINE
FRENCH
BALLENA
SPANISH
PEKEE
NUEE
FEGEE
ERROMANGOAN
EXTRACTS
Sub
Librarian
).
will

<Figure size 640x480 with 1 Axes>

低频词：
['Herman', 'Melville', ']', 'ETYMOLOGY', 'Late', 'Consumptive', 'School', 'threadbare', 'lexicons', 'mockingly', 'flags', 'mortality', 'signification', 'HACKLUYT', 'Sw', 'HVAL', 'roundness', 'Dut', 'Ger', 'WALLEN', 'WALW', 'IAN', 'RICHARDSON', 'KETOS', 'GREEK', 'CETUS', 'LATIN', 'WHOEL', 'ANGLO', 'SAXON', 'WAL', 'HWAL', 'SWEDISH', 'ICELANDIC', 'BALEINE', 'BALLENA', 'FEGEE', 'ERROMANGOAN', 'Librarian', 'painstaking', 'burrower', 'grub', 'Vaticans', 'stalls', 'higgledy', 'piggledy', 'gospel', 'promiscuously', 'commentator', 'belongest', 'sallow', 'Pale', 'Sherry', 'loves', 'bluntly', 'Subs', 'thankless', 'Hampton', 'Court', 'hie', 'refugees', 'pampered', 'Michael', 'Raphael', 'unsplinterable', 'GENESIS', 'JOB', 'JONAH', 'punish', 'ISAIAH', 'soever', 'cometh', 'incontinently', 'perisheth', 'PLUTARCH', 'MORALS', 'breedeth', 'Whirlpooles', 'Balaene', 'arpens', 'PLINY', 'Scarcely', 'TOOKE', 'LUCIAN', 'TRUE', 'catched', 'OCTHER', 'VERBAL', 'TAKEN', 'MOUTH', 'ALFRED', '890', 'gudgeon', 'r